In [ ]:
from axetract.preprocessor.axe_preprocessor import AXEPreprocessor

In [ ]:
dummy_html = """
<html>
    <head><title>Product Page</title></head>
    <body>
        <nav><ul><li>Home</li><li>Products</li></ul></nav>
        <div id="main-content" class="container" style="background: #fff;">
            <h1>SuperWidget 3000</h1>
            <p class="description">The SuperWidget 3000 is the best tool for <b>AI engineers</b>. 
            It features high-performance processing and low latency.</p>
            <table class="specs-table">
                <tr><th>Feature</th><th>Value</th></tr>
                <tr><td>Weight</td><td>1.2kg</td></tr>
                <tr><td>Price</td><td>$299</td></tr>
            </table>
            <div class="reviews">
                <h2>User Reviews</h2>
                <div class="review">"Life changing tool!" - Jane Doe</div>
                <div class="review">"Fast but expensive." - John Smith</div>
            </div>
        </div>
        <footer>Copyright 2025 WidgetCorp</footer>
    </body>
</html>
"""

# The query to use later with the LLM
query = "Extract the price and weight of the SuperWidget 3000 from the following context."

In [ ]:
from axetract.data_types import AXESample

test = AXESample(id="1", query=query, is_content_url=False, content=dummy_html)

In [ ]:
pre = AXEPreprocessor()
res = pre(test)
res

In [ ]:
res[0].chunks

In [ ]:
from axetract.pruner.axe_pruner import AXEPruner

In [ ]:
from axetract.llm.vllm_client import LocalVLLMClient
# from axetract.llm.hf_client import HuggingFaceClient

In [ ]:
lc = LocalVLLMClient(
    config={
        "model_name": "Qwen/Qwen3-0.6B",
        # api_key:"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",
        "max_tokens": 1024,  # max new tokens
        "engine_args": {
            "gpu_memory_utilization": 0.7,
            "max_model_len": 6000,  # max total tokens
            # "enforce_eager": True,
        },
        "temperature": 0.0,
        "lora_modules": {
            "pruner": {
                "path": "abdo-Mansour/Pruner_Adaptor_Qwen_3_FINAL_EXTRA",
                "temperature": 0.0,
            },
            "qa": {"path": "abdo-Mansour/Extractor_Adaptor_Qwen3_QA_websrc", "temperature": 1.0},
            "schema": {"path": "abdo-Mansour/Extractor_Adaptor_Qwen3_Final", "temperature": 0.0},
        },
    }
)

In [ ]:
from axetract.prompts.pruner_prompt import PRUNER_PROMPT
from axetract.prompts.qa_prompt import QA_PROMPT
from axetract.prompts.schema_prompt import SCHEMA_PROMPT

In [ ]:
pru = AXEPruner(
    llm_client=lc,
    llm_pruner_client=lc,
    llm_pruner_prompt=PRUNER_PROMPT,
)

In [ ]:
from axetract.extractor.axe_extractor import AXEExtractor

ext = AXEExtractor(
    llm_extractor_client=lc,
    schema_generation_prompt_template=SCHEMA_PROMPT,
    query_generation_prompt_template=QA_PROMPT,
)